# 🧠 Tutorial 2 — EEG Preprocessing
### 4th Year · Biomedical Signals & Applications · EEG Python Workshop

---

**Goal of this tutorial:** Clean the raw EEG so it is ready for analysis.

**By the end you will be able to:**
- Re-reference EEG to the average reference
- Apply notch and bandpass filters
- Detect and interpolate bad channels
- Run ICA to remove eye-blink and muscle artefacts
- Cut the continuous recording into epochs around motor imagery cues
- Apply baseline correction and reject noisy trials
- Save the clean epochs for Tutorial 3

**Time:** approximately 60 minutes

---

> ⚠️ **Run Tutorial 1 first** — it downloads the dataset that this notebook uses.

---
## Setup — Imports

Run this cell first every time you open the notebook.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import mne
from mne.datasets import eegbci
from mne.io import concatenate_raws, read_raw_edf
from mne.preprocessing import ICA

mne.set_log_level("WARNING")
print("MNE version:", mne.__version__)

---
## Step 0 — Load Raw Data

We reload the same motor imagery data from Tutorial 1.

The raw EEG object contains:
- **64 channels** × **~20,000 time points** = ~1.3 million numbers
- Two types of events: **T1** (left hand imagery) and **T2** (right hand imagery)
- Lots of noise — eye blinks, muscle tension, power-line interference

Our job in this tutorial is to clean all of that.

In [ ]:
subject = 1
runs    = [6, 10]

raw_files = eegbci.load_data(subject, runs, verbose=False)
raw = concatenate_raws([read_raw_edf(f, preload=True, verbose=False)
                        for f in raw_files])
eegbci.standardize(raw)
raw.set_montage(mne.channels.make_standard_montage("standard_1005"), verbose=False)

print("Raw data loaded")
print(f"  Channels : {len(raw.ch_names)}")
print(f"  Duration : {raw.times[-1]:.1f} s")
print(f"  Sfreq    : {raw.info['sfreq']} Hz")

---
## Step 1 — Re-Referencing

### What is a reference electrode?

Every EEG channel measures the **difference** in voltage between the active electrode and a reference electrode. Whatever signal exists at the reference gets subtracted from every channel — biasing all measurements simultaneously.

### Why re-reference?

The PhysioNet dataset was recorded against a single reference. If that reference electrode was near any brain activity, it contaminates all 64 channels at once. Switching to the **average reference** removes this bias:

```
V_cleaned(channel) = V_recorded(channel) − mean(all channels)
```

> 🟢 **Analogy:** Measuring the height of buildings relative to the top of one building (original reference) vs. relative to sea level (average reference). Sea level is more neutral.

> ⚠️ **Important:** Always fix bad channels *before* computing the average reference — a broken channel would corrupt the average and then corrupt everything else.

In [ ]:
raw_reref = raw.copy().set_eeg_reference("average", projection=False, verbose=False)
print("Re-referencing: set to average reference ✓")

# Visual comparison — one channel before vs after
data_before, times = raw.get_data(return_times=True)
data_after, _      = raw_reref.get_data(return_times=True)

fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
axes[0].plot(times[:1280], data_before[0, :1280] * 1e6, "steelblue", lw=0.8)
axes[0].set_title("Channel Fc5 — Before Re-referencing")
axes[0].set_ylabel("Amplitude (µV)")
axes[1].plot(times[:1280], data_after[0,  :1280] * 1e6, "tomato",    lw=0.8)
axes[1].set_title("Channel Fc5 — After Average Re-referencing")
axes[1].set_ylabel("Amplitude (µV)")
axes[1].set_xlabel("Time (s)")
plt.tight_layout()
plt.show()

> 👁 **What to check:**
> - The two traces should look nearly identical for this dataset (it was already average-referenced)
> - In real-world data with a biased reference you would see the DC level shift
> - The *shape* of the waveform should not change — only the overall level

---
## Step 2 — Filtering

### Why filter?

EEG contains unwanted signals at specific frequencies:
- **Very slow drifts (< 1 Hz):** Sweat, electrode movement — removed by **high-pass filter**
- **Power-line noise (60 Hz in USA, 50 Hz in Europe):** Electromagnetic interference from the electrical grid — removed by **notch filter**
- **High-frequency muscle noise (> 40 Hz):** Jaw and neck muscle activity — removed by **low-pass filter**

> 🟢 **Analogy:** A bandpass filter is like a window frame that only lets in light from one direction. We let in the frequencies between 1 and 40 Hz, and block everything outside that range.

| Filter type | What it removes | MNE command |
|---|---|---|
| Notch (60 Hz) | US power-line hum | `notch_filter(freqs=60)` |
| High-pass (1 Hz) | Slow drifts, sweat | `filter(l_freq=1.0, ...)` |
| Low-pass (40 Hz) | Muscle noise | `filter(..., h_freq=40.0)` |

> 💡 **European data?** Use `notch_filter(freqs=50)` instead of 60 Hz.

In [ ]:
raw_filt = raw_reref.copy()

# 2a. Notch filter — remove power-line interference
raw_filt.notch_filter(freqs=60, verbose=False)

# 2b. Bandpass filter — keep only 1–40 Hz
raw_filt.filter(l_freq=1.0, h_freq=40.0, verbose=False)

print("Filtering complete: notch at 60 Hz + bandpass 1–40 Hz ✓")

In [ ]:
# Compare PSD before vs after filtering
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

raw_reref.compute_psd(fmax=80).plot(axes=axes[0], show=False)
axes[0].set_title("PSD — Before Filtering")

raw_filt.compute_psd(fmax=80).plot(axes=axes[1], show=False)
axes[1].set_title("PSD — After Filtering")

plt.tight_layout()
plt.show()

> 👁 **What to check in the PSD plots:**
> - **Before:** Large hump at low frequencies + a sharp spike at 60 Hz
> - **After:** Clean spectrum that drops off above 40 Hz; the 60 Hz spike is gone
> - An alpha peak (~10 Hz) may be visible as a small bump

---
## Step 3 — Bad Channel Detection & Interpolation

### What is a bad channel?

In every real EEG session, some electrodes do not make good contact with the scalp. A bad electrode records noise instead of brain activity. Signs of a bad channel:
- **Flat line** — no signal at all (disconnected)
- **Extremely noisy** — 5–10× larger fluctuations than neighbours
- **Constant offset** — stuck at a different DC level

### The fix: interpolation

We estimate what the bad channel *should* have recorded based on the surrounding good channels. MNE uses **spherical spline interpolation** — fitting a smooth surface through all the good electrodes and reading off the value at the bad electrode's location.

> 🟢 **Analogy:** A bad channel is like a dead pixel on a screen. We estimate its correct colour from the surrounding pixels.

> ⚠️ **Rule:** Interpolate bad channels **before** re-referencing. A broken channel corrupts the average, which then corrupts all other channels.

In [ ]:
# For this demo, we manually mark Fp1 as bad
# In a real session: visually inspect raw.plot() and mark the ones you find
raw_bad = raw_filt.copy()
raw_bad.info["bads"] = ["Fp1"]
print("Marked as bad:", raw_bad.info["bads"])

# Interpolate the bad channel from its neighbours
raw_interp = raw_bad.copy().interpolate_bads(reset_bads=True, verbose=False)
print("After interpolation — bad channels:", raw_interp.info["bads"])  # should be empty

In [ ]:
# Visual comparison: original vs bad vs interpolated
idx_fp1 = raw_filt.ch_names.index("Fp1")
t_sel   = times[:640]  # first 4 seconds

data_orig   = raw_filt.get_data()[idx_fp1, :640]   * 1e6
data_interp = raw_interp.get_data()[idx_fp1, :640] * 1e6

fig, axes = plt.subplots(3, 1, figsize=(12, 6), sharex=True)
axes[0].plot(t_sel, data_orig,       "steelblue", lw=1, label="Original")
axes[0].set_title("Fp1 — Original (note: eye blink artefacts are normal here)")
axes[1].plot(t_sel, np.zeros(640),   "tomato",    lw=1, label="Zeroed (simulating dead channel)")
axes[1].set_title("Fp1 — Simulated dead channel (all zeros)")
axes[2].plot(t_sel, data_interp,     "seagreen",  lw=1, label="Interpolated")
axes[2].set_title("Fp1 — Reconstructed from neighbouring electrodes")

for ax in axes:
    ax.set_ylabel("µV")
    ax.legend(loc="upper right", fontsize=8)
axes[2].set_xlabel("Time (s)")
plt.tight_layout()
plt.show()

---
## Step 4 — ICA: Removing Physiological Artefacts

### The problem ICA solves

Eye blinks and muscle tension create electrical signals in the **same frequency range** as EEG (0–40 Hz). Simple filtering cannot remove them without also destroying the brain signal.

### How ICA works

ICA (Independent Component Analysis) mathematically separates the EEG into **independent sources**. Brain signals, eye blinks, and muscle activity are statistically independent — knowing when someone blinks tells you nothing about what their alpha rhythm is doing. ICA exploits this to disentangle them.

> 🟢 **Analogy:** The cocktail party problem. Three people talking simultaneously, three microphones in different positions — each microphone records a mixture of all three voices. ICA figures out what each individual person was saying by detecting that the three voices are independent of each other.

### The workflow
1. **Fit** ICA on the filtered data (learns 20 independent components)
2. **Identify** which components are artefacts (look at topography + time series)
3. **Exclude** the artefact components
4. **Apply** ICA to reconstruct clean electrode data

> ⚠️ **Golden rule:** When in doubt — **keep** the component. Removing a brain component permanently deletes neural information. Typically remove only 1–3 components (eye + possibly heartbeat).

In [ ]:
n_components = 20
ica = ICA(n_components=n_components, random_state=42, max_iter="auto")
ica.fit(raw_filt, verbose=False)

print(f"ICA fitted: {n_components} independent components extracted ✓")

In [ ]:
# Plot all component topographies — look for the frontal eye-blink blob
fig = ica.plot_components(show=True)
plt.suptitle("ICA Component Topographies — look for a large frontal blob (eye blink)")
plt.show()

> 👁 **What to look for in the topography plots:**
> - **Eye blink:** Large symmetric blob at the very front of the head (over Fp1/Fp2)
> - **Lateral eye movement:** Asymmetric front pattern — positive left, negative right (or vice versa)
> - **Muscle:** Focal spots at the edges/bottom of the scalp map
> - **Brain (alpha):** Smooth blob over the back of the head (occipital)
> - **Brain (motor mu):** Two symmetric spots over the C3/C4 area

In [ ]:
# Plot component time series — look for regular sharp triangular spikes (blinks)
fig = ica.plot_sources(raw_filt, show=True)
plt.show()

In [ ]:
# Automated eye artefact detection using Fp1 as a proxy for eye movements
eog_ch = "Fp1"
eog_idx, eog_scores = ica.find_bads_eog(raw_filt, ch_name=eog_ch, verbose=False)
print(f"Suspected eye-movement components: {eog_idx}")

# Plot correlation scores
fig = ica.plot_scores(eog_scores, exclude=eog_idx, show=True,
                      title="ICA component correlation with Fp1 (eye proxy)")
plt.show()

In [ ]:
# Exclude the identified artefact components and reconstruct clean EEG
ica.exclude = eog_idx
raw_ica = raw_filt.copy()
ica.apply(raw_ica, verbose=False)

print(f"Artefact components removed: {ica.exclude}")
print(f"Clean data shape: {raw_ica.get_data().shape}")

> 👁 **What to check after ICA:**
> - Replot the raw data — the large frontal spikes (eye blinks) should be much smaller or gone
> - The rest of the signal should look broadly similar — we only removed artefact sources
> - If the signal looks completely different, you may have removed too many components

---
## Step 5 — Epoching

### From continuous recording to trials

So far we have one long recording. But our experiment had many individual trials — each time the subject received a cue (T1 or T2) to imagine moving their hand. **Epoching** cuts the continuous EEG into fixed-length windows aligned to each event.

Each epoch runs from **−0.5 s** (before the cue, for baseline) to **+3.0 s** (end of the imagery period):

```
|--baseline--|------ imagery period --------|
-0.5s        0s                            +3.0s
              ↑ cue appears here
```

### Baseline correction

After cutting epochs, we subtract the mean voltage during the pre-cue period (−0.5 to 0 s) from the whole epoch. This zeros out any slow drift that happened to be present at the moment the cue appeared.

> 🟢 **Analogy:** Measuring how fast runners improve over a 100 m sprint. You don't care where they started — you zero their starting position and only measure the *change*.

In [ ]:
# Extract event markers (T1 = left hand, T2 = right hand)
events, event_id = mne.events_from_annotations(raw_ica, verbose=False)
print("Event IDs:", event_id)

# Keep only motor imagery events
event_id_sel = {"left_hand": event_id["T1"], "right_hand": event_id["T2"]}

tmin, tmax = -0.5, 3.0   # epoch window

epochs = mne.Epochs(
    raw_ica,
    events,
    event_id   = event_id_sel,
    tmin       = tmin,
    tmax       = tmax,
    baseline   = (None, 0),   # baseline-correct using the pre-stimulus interval
    preload    = True,
    verbose    = False,
)

print(f"\nEpochs created:")
print(f"  Trials   : {len(epochs)}")
print(f"  Shape    : {epochs.get_data().shape}  (trials × channels × time points)")
print(f"  Conditions: {list(epochs.event_id.keys())}")

In [ ]:
# Plot the first 5 epochs at channel C3
ep_data  = epochs.get_data()
ep_times = epochs.times
idx_c3   = epochs.ch_names.index("C3")

fig, ax = plt.subplots(figsize=(11, 3))
for i in range(min(5, len(ep_data))):
    ax.plot(ep_times, ep_data[i, idx_c3] * 1e6, alpha=0.6, lw=0.9,
            label=f"Trial {i+1}")
ax.axvline(0, color="k", lw=1.5, linestyle="--", label="Cue onset (t=0)")
ax.axvspan(tmin, 0, alpha=0.08, color="grey", label="Baseline period")
ax.set_xlabel("Time (s)")
ax.set_ylabel("Amplitude (µV)")
ax.set_title("C3 — First 5 Epochs (each line is one trial)")
ax.legend(fontsize=8, loc="upper right")
plt.tight_layout()
plt.show()

In [ ]:
# Epoch image — rows=trials, columns=time, colour=amplitude
# This gives an overview of ALL trials at once
fig = epochs.plot_image(picks=["C3"], show=True)
plt.show()

---
## Step 6 — Epoch Rejection

Even after ICA, some trials may still contain extreme-amplitude events (e.g., a sudden jaw clench). We reject any epoch where any channel exceeds a threshold of **±150 µV**.

| Threshold | Effect |
|---|---|
| Too strict (50 µV) | Many epochs rejected; too few trials for reliable analysis |
| Too lenient (500 µV) | Artefact-contaminated trials survive; corrupt results |
| Typical (100–150 µV) | Good balance for adult subjects |

**Aim to retain at least 70% of epochs.** If you are losing more, something in the earlier steps may need adjustment.

In [ ]:
n_before = len(epochs)

epochs_clean = epochs.copy().drop_bad(
    reject={"eeg": 150e-6},   # ±150 µV threshold
    verbose=False,
)

n_after  = len(epochs_clean)
retained = n_after / n_before * 100

print(f"Epochs before rejection : {n_before}")
print(f"Epochs after  rejection : {n_after}")
print(f"Retained                : {retained:.1f}%")

if retained < 70:
    print("⚠️  Less than 70% retained — consider adjusting the threshold or reviewing ICA")
else:
    print("✓  Retention rate is acceptable")

---
## Step 7 — Save Clean Epochs

We save the preprocessed epochs in MNE's `.fif` format. Tutorial 3 will load this file directly — no need to re-run all the preprocessing.

In [ ]:
# Apply explicit baseline correction (already applied in Epochs, but shown for clarity)
epochs_bc = epochs_clean.copy().apply_baseline((None, 0), verbose=False)

save_path = "/tmp/motor_imagery_epochs-epo.fif"
epochs_bc.save(save_path, overwrite=True, verbose=False)

print(f"Clean epochs saved to: {save_path} ✓")
print(f"Final shape: {epochs_bc.get_data().shape}  (trials × channels × time points)")

# Verify we can load them back
epochs_loaded = mne.read_epochs(save_path, verbose=False)
print(f"Load check: {epochs_loaded.get_data().shape} ✓")

---
## ✅ Tutorial 2 — Summary

| Step | MNE Command | What it does |
|---|---|---|
| 1 — Re-reference | `set_eeg_reference('average')` | Remove reference electrode bias |
| 2 — Notch filter | `notch_filter(freqs=60)` | Delete 60 Hz power-line hum |
| 2 — Bandpass | `filter(l_freq=1, h_freq=40)` | Keep only 1–40 Hz |
| 3 — Bad channels | `info['bads'] = [...]` | Mark broken electrodes |
| 3 — Interpolate | `interpolate_bads()` | Reconstruct from neighbours |
| 4 — ICA fit | `ICA().fit(raw)` | Learn 20 independent components |
| 4 — ICA identify | `find_bads_eog()` + visual | Find artefact components |
| 4 — ICA apply | `ica.apply(raw)` | Remove artefacts; reconstruct EEG |
| 5 — Epoch | `mne.Epochs(tmin, tmax, baseline)` | Cut into 3.5 s windows |
| 6 — Reject | `epochs.drop_bad(reject={...})` | Discard trials above threshold |
| 7 — Save | `epochs.save('-epo.fif')` | Store to disk for Tutorial 3 |

---

> ⚠️ **Order matters!**
> `Fix bad channels → Re-reference → Filter → ICA → Epoch`
> Changing the order leads to worse artefact removal.

---

### ➜ Next: Tutorial 3 — EEG Analysis & Classification

In Tutorial 3 we will use these clean epochs to:
- Compute ERPs and compare left vs right hand conditions
- Analyse the frequency spectrum and band power
- Build time-frequency maps showing ERD/ERS
- Extract features and train a left/right classifier